In [ ]:
# Install dependencies (run once per fresh environment, before any imports;
# restart the kernel afterwards if packages were upgraded)
%pip install -q -r ../requirements.txt

In [ ]:
# --- Setup (Step 0) ---
import sys
from pathlib import Path

# Make src/ importable whether run locally (cwd=src/) or from repo root
for p in (Path.cwd(), Path.cwd() / "src"):
    if (p / "config.py").is_file() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

import config as C

C.set_seed()  # Task 2 runs no model, so no Hugging Face token is needed
print("repo root:", C.REPO_ROOT)

# Task 2 – RQ2: How do reasoning-trace characteristics relate to performance?

Gemma 4 E4B, the reasoning model from Task 1, writes out its reasoning (a "trace") before every answer. Task 1 saved all 1034 traces. This notebook measures **two features** of each trace and tests how they relate to (a) whether the answer was **correct** and (b) the question's **difficulty**.

| | Feature | What it measures | Why it matters for Text-to-SQL |
|---|---|---|---|
| 1 | **Trace length** (thinking tokens) | How much the model reasons before answering | Harder SQL (more joins, conditions, nesting) may need more planning; very long reasoning may also signal that the model is stuck or unsure |
| 2 | **Number of SQL drafts** | How many SQL queries the model writes out while reasoning | Shows whether the model builds the query in pieces or rewrites it; many drafts may mean revising and searching for the right query |

**Correctness** is execution accuracy (EX) from Task 1: the predicted query returns the same rows as the gold query. We use EX and not exact match (EM), because for E4B the official EM parser rejects many correct queries (see Task 1), so EM would mislabel them as wrong.

**Important: difficulty affects everything.** Harder questions get longer traces and more drafts, *and* they are answered wrongly more often. So a simple "wrong answers have longer traces" could just mean "hard questions are hard". Every test below therefore either compares correct and wrong answers **within the same difficulty level**, or **holds difficulty fixed** in a model.

The step-1 cells extract the features; the step-2 cells compute descriptive statistics, run the tests and draw the plots. All tables are saved to `outputs/task2/metrics/task2_*.csv` and the figures to `outputs/task2/figures/task2_*`. No GPU is needed; the notebook runs in seconds.

## Step 1.1 – Load the traces and join them with correctness and difficulty

This cell reads E4B's full-run outputs (`outputs/gemma4-e4b/baseline/responses.jsonl`, one record per example with its reasoning trace in `thinking`) and Task 1's per-example scores (`outputs/task1/metrics/task1_per_example.csv`), and matches them by example `id`. It checks that there is exactly one trace for each of the 1034 dev examples, that no trace is empty, and that no output was cut off at the token limit (a cut-off trace would make its length meaningless).

In [ ]:
import json

import numpy as np
import pandas as pd

E4B_KEY = next(key for key, spec in C.MODELS.items() if spec.reasoning)  # the only model with traces
E4B = C.MODELS[E4B_KEY].name
LEVELS = ["easy", "medium", "hard", "extra"]

inputs = {ex["id"]: ex for ex in json.loads(C.DEV_INPUTS.read_text())}
records = [json.loads(line) for line in (C.run_path(E4B_KEY) / "responses.jsonl").read_text().splitlines()]
traces = pd.DataFrame({"id": [r["id"] for r in records],
                       "thinking": [r["thinking"] for r in records],
                       "thinking_tokens": [r["thinking_tokens"] for r in records],
                       "truncated": [r["truncated"] for r in records]})

scores = pd.read_csv(C.TASK1_METRICS_DIR / "task1_per_example.csv")
scores = scores.loc[scores["model"] == E4B, ["id", "difficulty", "ex", "em"]]

df = traces.merge(scores, on="id", validate="one_to_one").sort_values("id").reset_index(drop=True)
df["correct"] = df["ex"].astype(int)  # 1 = EX-correct, 0 = wrong

assert len(df) == len(inputs) and set(df["id"]) == set(inputs), "traces don't match the dev examples"
assert df["thinking"].str.strip().ne("").all(), "empty trace"
assert not df["truncated"].any(), "a truncated trace would make its length meaningless"
print(f"{len(df)} traces from {E4B}; EX-correct: {df['correct'].sum()} ({df['correct'].mean():.1%})")
print(pd.crosstab(df["difficulty"], df["correct"].map({1: "correct", 0: "wrong"})).reindex(LEVELS))

## Step 1.2 – Feature 1: trace length

**Definition:** the number of tokens in the reasoning trace, as counted by E4B's own tokenizer in Task 1 (`thinking_tokens`). A token is a small piece of text, roughly ¾ of a word on average.

**Why it is relevant:** it measures how much the model "thinks" before writing its final SQL. For Text-to-SQL, a question that needs several tables, conditions or a nested query may need more planning; but a very long trace can also mean the model is going back and forth.

We also keep the length in characters as a cross-check: the two measures should agree closely.

In [ ]:
df["trace_chars"] = df["thinking"].str.len()
rho = df[["thinking_tokens", "trace_chars"]].corr(method="spearman").iloc[0, 1]
print(f"trace length: median {df['thinking_tokens'].median():.0f} tokens, range {df['thinking_tokens'].min()}-{df['thinking_tokens'].max()}")
print(f"cross-check: tokens vs. characters, Spearman correlation {rho:.3f} (close to 1 = same ordering)")

## Step 1.3 – Feature 2: number of SQL drafts

**Definition:** the number of **distinct SQL queries the model writes out in its trace**. A piece of text counts as a draft if it is a query, i.e. it contains both `SELECT` and `FROM`, written in any of the three ways E4B uses:
1. in a code block (` ```sql ... ``` `);
2. as inline code (`` `SELECT ... FROM ...` ``);
3. clause by clause on consecutive lines (e.g. a numbered list `1. SELECT Name`, `2. FROM teacher`, …), which are joined into one query.

Queries are compared after ignoring spaces, line breaks and upper/lower case, so writing the same query twice counts once. Sub-queries and rewritten versions count as separate drafts. Single keywords or column names (`COUNT(*)`, "**SELECT** the name") are not drafts. In plain text, only uppercase SQL keywords count, because E4B also uses "Select … from" as ordinary English.

**Why it is relevant:** it shows *how* the model builds its query: written once, built from smaller pieces, or rewritten after noticing a problem. For example, in example 334 the model writes a query with `JOIN`, notices it would miss template types with no templates ("Wait, if I use an INNER JOIN…"), and rewrites it with `LEFT JOIN` (2 drafts). In example 214 it writes one query and answers with it (1 draft). In example 782 it writes two sub-queries and then combines them (3 drafts).

**Checking the counter:** the counting rule was compared with counts made by hand on 24 traces (20 random traces plus 4 worked examples) and agreed on all 24. The `assert` below repeats four of these checks.

In [ ]:
import re

FENCED = re.compile(r"```[^\n`]*\n(.*?)```", re.S)  # code blocks
CLAUSE = r"(SELECT|FROM|WHERE|JOIN|INNER|LEFT|GROUP|ORDER|HAVING|LIMIT|UNION|EXCEPT|INTERSECT|AND|OR|ON)\b"
CLAUSE_ANY_CASE = re.compile(r"^\s*" + CLAUSE, re.I)  # inside code, any case counts
CLAUSE_UPPER = re.compile(r"^\s*" + CLAUSE)  # in plain text, only UPPERCASE keywords count


def is_query(text):
    return bool(re.search(r"\bSELECT\b", text, re.I) and re.search(r"\bFROM\b", text, re.I))


def normalize(query):
    """Ignore spacing, line breaks, case and semicolons when comparing queries."""
    return " ".join(query.replace(";", " ").replace("(", " ( ").replace(")", " ) ").split()).lower()


def sql_drafts(trace):
    """Return the distinct SQL queries written out in a trace, in order of appearance."""
    # 1) queries inside code blocks
    found = [(m.start(), m.group(1)) for m in FENCED.finditer(trace) if is_query(m.group(1))]
    # 2) + 3) inline code and plain lines: collect consecutive SQL clause lines into one query
    text = FENCED.sub(lambda m: "\n" * m.group(0).count("\n"), trace)  # blank out code blocks (already counted)
    run, run_start, pos = [], 0, 0

    def close_run():
        if run and is_query(" ".join(run)):
            found.append((run_start, " ".join(run)))
        run.clear()

    for line in text.split("\n"):
        inline = [s for s in re.findall(r"`([^`\n]+)`", line) if CLAUSE_ANY_CASE.match(s)]
        plain = re.sub(r"^\s*(\d+\.|[-*])\s*", "", line)  # drop list markers like "1." or "-"
        if inline:
            part = " ".join(inline)
        elif CLAUSE_UPPER.match(plain) and not re.search(r"\*\*|:\s*$", plain):  # skip bold labels / headings
            part = plain.strip()
        else:
            part = None
        if part is not None:
            if re.match(r"\s*SELECT\b", part, re.I) and run:  # a new SELECT starts a new query
                close_run()
            if not run:
                run_start = pos
            run.append(part)
        elif line.strip() or run:  # any other text (or a blank line after SQL) ends the current query
            close_run()
        pos += len(line) + 1
    close_run()
    # keep each distinct query once, in order of appearance
    distinct, seen = [], set()
    for _, query in sorted(found):
        if normalize(query) not in seen:
            seen.add(normalize(query))
            distinct.append(query)
    return distinct


df["n_drafts"] = df["thinking"].map(lambda t: len(sql_drafts(t)))

# hand-checked examples (see markdown above)
by_id = df.set_index("id")["n_drafts"]
assert by_id[334] == 2 and by_id[214] == 1 and by_id[782] == 3 and by_id[487] == 3, by_id[[334, 214, 782, 487]].to_dict()
print("draft counts over all traces:", df["n_drafts"].value_counts().sort_index().to_dict())

# show what the counter finds in a few traces, for transparency
for i in (334, 782):
    print(f"\nexample {i} ({inputs[i]['difficulty']}): {inputs[i]['question']}")
    for k, q in enumerate(sql_drafts(df.loc[df["id"] == i, "thinking"].item()), 1):
        print(f"  draft {k}: {' '.join(q.split())[:140]}")

## Step 1.4 – Save the features

This cell saves one row per example to `outputs/task2/metrics/task2_features.csv`: `id`, `difficulty`, `correct` (EX), `thinking_tokens`, `trace_chars` and `n_drafts`. The trace text itself is not copied, because it is already in the model's output file. All Step 2 statistics are computed from this table.

In [ ]:
features = df[["id", "difficulty", "correct", "thinking_tokens", "trace_chars", "n_drafts"]]
features.to_csv(C.TASK2_METRICS_DIR / "task2_features.csv", index=False)
print("saved", C.TASK2_METRICS_DIR / "task2_features.csv", features.shape)

# Step 2 – Statistics: how do the features relate to difficulty and correctness?

We use standard statistical tests that fit the shape of our data. They are our choice; the assignment does not prescribe them. Two facts about the data drive the choice:
- **The features are not bell-shaped.** Trace lengths are skewed: most traces are short, a few are very long. The draft counts are small whole numbers (mostly 1). Tests that assume normally distributed data (t-test, ANOVA) don't fit, so we use **rank-based** tests. These sort all values from smallest to largest and work with the positions, so a few extreme values can't dominate.
- **Difficulty is a confounder.** It raises both the features and the error rate, so it must be controlled for.

Which test answers which question, and on which measure:

| Step | Test | Question it answers | Applied to | Grouping / outcome | Why this test |
|---|---|---|---|---|---|
| 2.2 | **Kruskal–Wallis** | Does the feature differ between difficulty levels? | trace length; number of drafts | 4 difficulty levels | Compares more than two groups using ranks; no normality assumption |
| 2.2 | **Spearman correlation** | Does the feature rise steadily with difficulty, and how strongly? | trace length; number of drafts | difficulty coded 1–4 (easy → extra) | Rank-based measure of a monotonic trend; difficulty is ordered |
| 2.3 | **Mann–Whitney U** + rank-biserial *r* | Within one difficulty level, do correct and wrong answers differ on the feature? | trace length; number of drafts | EX-correct vs. EX-wrong, within each level | Compares two groups using ranks; testing *within* a level removes the difficulty effect |
| 2.3 | **Holm correction** | Is a difference still significant after running many tests? | the 8 Mann–Whitney p-values (2 features × 4 levels) | – | Running many tests raises the chance of a false "significant" result; Holm adjusts for it |
| 2.4 | **Logistic regression** | At the *same* difficulty, does the feature change the chance of a correct answer, and by how much? | trace length (log scale); number of drafts | outcome: EX-correct (yes/no); difficulty as a control | The outcome is yes/no; one model uses all 1034 examples while holding difficulty fixed, and can include both features to see whether each adds information |

A result is called **significant** when its p-value is below 0.05: a difference that large would be unlikely (under 5% chance) if there were really no difference.

## Step 2.1 – Descriptive statistics by correctness and difficulty

This table describes each feature for every combination of **difficulty level** (easy, medium, hard, extra, and all together) and **correctness** (EX-correct or wrong):
- `n`: the number of examples;
- the mean and standard deviation (SD);
- the median and the interquartile range (IQR, the middle 50% of values, from the 25th to the 75th percentile);
- the minimum and maximum.

For the number of drafts we also report the share of traces with two or more drafts. Saved to `outputs/task2/metrics/task2_descriptive.csv`.

In [ ]:
FEATURES = {"thinking_tokens": "trace length (thinking tokens)", "n_drafts": "number of SQL drafts"}
CORRECTNESS = {1: "correct", 0: "wrong"}

rows = []
for feature in FEATURES:
    for level in LEVELS + ["all"]:
        in_level = df if level == "all" else df[df["difficulty"] == level]
        for flag, label in CORRECTNESS.items():
            x = in_level.loc[in_level["correct"] == flag, feature]
            rows.append({"feature": feature, "difficulty": level, "correctness": label, "n": len(x),
                         "mean": round(x.mean(), 2), "sd": round(x.std(), 2), "median": x.median(),
                         "q1": x.quantile(0.25), "q3": x.quantile(0.75), "min": x.min(), "max": x.max(),
                         "share_2plus_drafts": round((x >= 2).mean(), 3) if feature == "n_drafts" else None})
descriptive = pd.DataFrame(rows)
assert descriptive.groupby("feature")["n"].sum().eq(2 * len(df)).all()  # each example counted once per level + once in "all"
descriptive.to_csv(C.TASK2_METRICS_DIR / "task2_descriptive.csv", index=False)

for feature, label in FEATURES.items():
    print(f"\n{label}: median (mean) by difficulty and correctness")
    d = descriptive[descriptive["feature"] == feature]
    show = d.assign(value=d.apply(lambda r: f"{r['median']:g} ({r['mean']:.1f}), n={r['n']}", axis=1))
    display(show.pivot(index="difficulty", columns="correctness", values="value").reindex(LEVELS + ["all"]))

## Step 2.2 – Feature vs. difficulty: Kruskal–Wallis and Spearman correlation

**Question:** does each feature change with the difficulty of the question?

- **Kruskal–Wallis test**, applied to **trace length** and to **number of drafts** separately, with the 4 difficulty levels as groups. It pools all traces, ranks them from smallest to largest feature value, and checks whether the average rank differs between the levels more than chance would explain. It is the rank-based version of a one-way ANOVA, chosen because our features are skewed or discrete. A small p-value means at least one level differs.
- **Spearman correlation (ρ)** between each feature and difficulty coded 1 = easy, 2 = medium, 3 = hard, 4 = extra. Kruskal–Wallis only says *whether* the levels differ. Spearman says *in which direction* and *how strongly* the feature changes as difficulty increases: from −1 (steadily falls) through 0 (no trend) to +1 (steadily rises). It is rank-based, so it only assumes the trend goes one way, not that it is a straight line.

All examples are used here, correct and wrong together, because the question is about difficulty only. Results are saved to `outputs/task2/metrics/task2_tests.csv`.

In [ ]:
from scipy import stats

df["difficulty_code"] = df["difficulty"].map({lvl: k for k, lvl in enumerate(LEVELS, 1)})  # easy=1 ... extra=4

difficulty_tests = []
for feature, label in FEATURES.items():
    # Kruskal-Wallis: one group of feature values per difficulty level.
    # Internally it ranks all 1034 values together, averages the ranks within each level, and measures how far
    # those average ranks are from what we'd expect if difficulty made no difference (statistic H).
    groups = [df.loc[df["difficulty"] == level, feature] for level in LEVELS]
    h, p_kw = stats.kruskal(*groups)
    # Spearman: rank the feature values and the difficulty codes, then correlate the ranks.
    # rho > 0 means the feature tends to be larger for harder questions.
    rho, p_rho = stats.spearmanr(df[feature], df["difficulty_code"])
    difficulty_tests.append({"feature": feature, "test": "Kruskal-Wallis (4 difficulty levels)", "statistic": round(h, 3), "p_value": p_kw})
    difficulty_tests.append({"feature": feature, "test": "Spearman rho with difficulty (1-4)", "statistic": round(rho, 3), "p_value": p_rho})
    print(f"{label}: Kruskal-Wallis H = {h:.1f}, p = {p_kw:.2g}; Spearman rho = {rho:.3f}, p = {p_rho:.2g}")
difficulty_tests = pd.DataFrame(difficulty_tests)

## Step 2.3 – Feature vs. correctness within each difficulty level: Mann–Whitney U

**Question:** for questions of the *same* difficulty, do correct and wrong answers differ in trace length or number of drafts?

- **Mann–Whitney U test**, applied to **trace length** and to **number of drafts** separately. It compares two groups, **EX-correct vs. EX-wrong** answers, **within one difficulty level at a time**: it is run 4 times per feature, once for easy, medium, hard and extra. Comparing within a level is how we remove the effect of difficulty: all the questions being compared are of the same difficulty, so difficulty cannot explain a difference. The test ranks all the values in the level and checks whether the wrong answers tend to sit higher (or lower) in that ranking than the correct ones. It is the rank-based version of a two-sample t-test, chosen because our features are skewed or discrete.
- **Effect size: rank-biserial correlation (r).** A p-value says whether a difference is likely to be real, not how big it is. The rank-biserial r answers "how big": if you pick one wrong and one correct answer at random from the same level, r is (chance the wrong one has the larger value) − (chance it has the smaller value). r = 0 means no difference; r = +0.3 means the wrong answer has the larger value noticeably more often; the range is −1 to +1. Rough guide: 0.1 small, 0.3 medium, 0.5 large.
- **Holm correction.** We run 8 of these tests (2 features × 4 levels). With many tests, one may look significant by chance. Holm's method makes the threshold stricter: it sorts the p-values from smallest to largest and multiplies the smallest by 8, the next by 7, and so on, keeping the order consistent. We report both the raw and the Holm-adjusted p-value, and call a result significant only if the **adjusted** p-value is below 0.05.

For reference, the same test is also shown for all levels pooled ("all"). That comparison is mixed up with difficulty, so it is not used for conclusions and is left out of the Holm correction. Results are saved to `outputs/task2/metrics/task2_tests.csv`.

In [ ]:
from statsmodels.stats.multitest import multipletests

within = []
for feature, label in FEATURES.items():
    for level in LEVELS + ["all"]:
        in_level = df if level == "all" else df[df["difficulty"] == level]
        wrong = in_level.loc[in_level["correct"] == 0, feature]
        right = in_level.loc[in_level["correct"] == 1, feature]
        # Mann-Whitney U: rank all values in this level together; U counts, over every (wrong, correct) pair,
        # how often the wrong answer's value is larger (ties count as half).
        u, p = stats.mannwhitneyu(wrong, right, alternative="two-sided")
        pairs = len(wrong) * len(right)
        # rank-biserial r = P(wrong larger) - P(wrong smaller) = 2 * U / (number of pairs) - 1
        r = 2 * u / pairs - 1
        within.append({"feature": feature, "difficulty": level, "n_wrong": len(wrong), "n_correct": len(right),
                       "median_wrong": wrong.median(), "median_correct": right.median(),
                       "U": u, "p_value": p, "rank_biserial_r": round(r, 3)})
within = pd.DataFrame(within)

# Holm correction over the 8 within-level tests (the pooled "all" rows are reference only)
per_level = within["difficulty"] != "all"
reject, p_holm, _, _ = multipletests(within.loc[per_level, "p_value"], alpha=0.05, method="holm")
within.loc[per_level, "p_holm"] = p_holm
within.loc[per_level, "significant_after_holm"] = reject
display(within.round({"p_value": 4, "p_holm": 4}))

## Step 2.4 – One model holding difficulty fixed: logistic regression

**Question:** at the *same* difficulty, how does each feature change the chance that the answer is correct, and does the number of drafts tell us anything beyond trace length?

**Logistic regression** predicts a yes/no outcome, here **EX-correct (1) or wrong (0)**, from one or more inputs. We fit three models on all 1034 examples:
1. correct ~ **trace length** + difficulty
2. correct ~ **number of drafts** + difficulty
3. correct ~ **trace length** + **number of drafts** + difficulty

Difficulty is included in every model as a **control**: each model estimates the feature's effect *as if all questions had the same difficulty*. Easy is the reference level, and medium, hard and extra get their own adjustment. Unlike the within-level tests (Step 2.3), which use each level separately, one model uses all the data at once and gives one overall number per feature. Model 3 checks whether each feature still matters once the other is known, which is useful because more drafts usually also means a longer trace.

**How to read the result:** the model reports an **odds ratio (OR)** per feature. Odds are the chance of a correct answer divided by the chance of a wrong one (90% correct = odds of 9). OR = 1 means the feature makes no difference. OR = 0.5 means the odds of being correct are **halved**, and OR = 2 means they double.
- **Trace length** enters as log₂(tokens), so its OR means the change **per doubling of the trace length** (e.g. 200 → 400 tokens). Lengths are very skewed, and a doubling is a natural step.
- **Drafts** enter as a count, so their OR is the change **per extra draft**.

Each OR comes with a 95% confidence interval (CI): the range of values consistent with the data. If the CI does not include 1, the effect is significant at the 0.05 level. We also report the **Spearman correlation between the two features**, to show how much they overlap. Results are saved to `outputs/task2/metrics/task2_logistic.csv`.

In [ ]:
import statsmodels.formula.api as smf

df["log2_tokens"] = np.log2(df["thinking_tokens"])  # 1 unit = a doubling of the trace length
# Difficulty as a control: a categorical variable whose first category (easy) is the reference level, so the model
# estimates one adjustment each for medium, hard and extra. (We avoid patsy's C(...) here, because C is our config module.)
data = df.assign(difficulty=pd.Categorical(df["difficulty"], categories=LEVELS))
models = {"1: length": "correct ~ log2_tokens + difficulty",
          "2: drafts": "correct ~ n_drafts + difficulty",
          "3: length + drafts": "correct ~ log2_tokens + n_drafts + difficulty"}
TERMS = {"log2_tokens": "trace length (per doubling)", "n_drafts": "number of drafts (per extra draft)"}

logistic = []
for name, formula in models.items():
    # Fit by maximum likelihood: find the coefficients that make the observed correct/wrong outcomes most likely.
    fit = smf.logit(formula, data=data).fit(disp=0)
    assert fit.mle_retvals["converged"], f"model {name} did not converge"
    ci = fit.conf_int()  # 95% confidence interval of each coefficient (on the log-odds scale)
    for term, label in TERMS.items():
        if term in fit.params:
            # coefficient = change in log-odds; exp(coefficient) = odds ratio (OR)
            logistic.append({"model": name, "term": label, "odds_ratio": round(np.exp(fit.params[term]), 3),
                             "ci_low": round(np.exp(ci.loc[term, 0]), 3), "ci_high": round(np.exp(ci.loc[term, 1]), 3),
                             "p_value": fit.pvalues[term], "n": int(fit.nobs),
                             "pseudo_r2": round(fit.prsquared, 3)})  # McFadden's pseudo-R^2: share of "explained" fit
logistic = pd.DataFrame(logistic)
logistic.to_csv(C.TASK2_METRICS_DIR / "task2_logistic.csv", index=False)
display(logistic.round({"p_value": 6}))

# how much do the two features overlap?
rho_ft, p_ft = stats.spearmanr(df["thinking_tokens"], df["n_drafts"])
print(f"Spearman correlation between trace length and number of drafts: rho = {rho_ft:.3f} (p = {p_ft:.2g})")

# all hypothesis tests in one file
tests = pd.concat([difficulty_tests.assign(difficulty="all levels"),
                   within.assign(test="Mann-Whitney U (wrong vs. correct)", statistic=within["U"])
                         .drop(columns=["U"]),
                   pd.DataFrame([{"feature": "thinking_tokens vs n_drafts", "test": "Spearman rho between features",
                                  "statistic": round(rho_ft, 3), "p_value": p_ft, "difficulty": "all levels"}])],
                  ignore_index=True)
tests.to_csv(C.TASK2_METRICS_DIR / "task2_tests.csv", index=False)
print("saved", C.TASK2_METRICS_DIR / "task2_tests.csv", "and task2_logistic.csv")

## Step 2.5 – Plots

This cell draws two figures and saves each as PNG and PDF in `outputs/task2/figures/`:
1. **`task2_feature_boxplots`:** box plots of each feature (left: trace length, on a log scale because it is very skewed; right: number of drafts) for every difficulty level, with **correct** (blue) and **wrong** (orange) answers side by side.
   - Each box spans the middle 50% of the values, the line inside it is the median, and the whiskers reach the furthest values within 1.5 × the box height.
   - The **mean** is marked with a white diamond. This matters for the number of drafts: its values are small whole numbers, so many boxes share the same median (1), and the mean shows differences the median hides.
2. **`task2_accuracy_by_drafts`:** the share of EX-correct answers among traces with 0, 1, 2, and 3 or more drafts, with the number of traces in each group. This one does not control for difficulty; the tests above do.

In [ ]:
import matplotlib.pyplot as plt

GROUP_COLORS = {"correct": "#2a78d6", "wrong": "#eb6834"}  # validated categorical palette, slots 1-2
E4B_COLOR = "#1baf7a"  # E4B's color in all Task 1 figures
INK, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#e1e0d9", "#c3c2b7"
plt.rcParams.update({"font.size": 10, "axes.edgecolor": AXIS, "axes.labelcolor": MUTED, "xtick.color": MUTED,
                     "ytick.color": MUTED, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110,
                     "pdf.fonttype": 42})
C.TASK2_FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save(fig, name):
    fig.savefig(C.TASK2_FIGURES_DIR / f"{name}.png", dpi=200, bbox_inches="tight")
    fig.savefig(C.TASK2_FIGURES_DIR / f"{name}.pdf", bbox_inches="tight", metadata={"CreationDate": None})
    plt.show()


# 1. box plots by difficulty x correctness
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
for ax, (feature, label) in zip(axes, FEATURES.items()):
    for k, level in enumerate(LEVELS):
        for j, (flag, group) in enumerate([(1, "correct"), (0, "wrong")]):
            x = df.loc[(df["difficulty"] == level) & (df["correct"] == flag), feature]
            pos = k + (-0.2 if j == 0 else 0.2)
            ax.boxplot(x, positions=[pos], widths=0.34, patch_artist=True, showfliers=True,
                       boxprops={"facecolor": GROUP_COLORS[group], "edgecolor": GROUP_COLORS[group], "alpha": 0.85},
                       medianprops={"color": "white", "linewidth": 1.6},
                       whiskerprops={"color": GROUP_COLORS[group]}, capprops={"color": GROUP_COLORS[group]},
                       flierprops={"marker": "o", "markersize": 2.5, "markerfacecolor": GROUP_COLORS[group],
                                   "markeredgecolor": "none", "alpha": 0.5})
            ax.scatter(pos, x.mean(), marker="D", s=26, color="white", edgecolor=INK, linewidth=0.8, zorder=4)
            ax.annotate(f"{x.mean():.0f}" if feature == "thinking_tokens" else f"{x.mean():.2f}",
                        (pos, x.mean()), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=7, color=MUTED)
    ax.set_xticks(range(len(LEVELS)), [f"{lvl}\n(n={(df['difficulty'] == lvl).sum()})" for lvl in LEVELS])
    ax.set_title(label, color=INK, fontsize=11, loc="left")
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
axes[0].set_yscale("log")
axes[0].set_yticks([50, 100, 200, 500, 1000, 2000], ["50", "100", "200", "500", "1000", "2000"])
axes[0].minorticks_off()
axes[0].set_ylabel("thinking tokens (log scale)")
axes[1].set_ylabel("SQL drafts in the trace")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in GROUP_COLORS.values()]
handles.append(plt.Line2D([], [], marker="D", color="white", markeredgecolor=INK, linestyle="none"))
axes[1].legend(handles, ["EX-correct", "EX-wrong", "mean"], frameon=False, fontsize=9, loc="upper left")
fig.tight_layout()
save(fig, "task2_feature_boxplots")

# 2. accuracy by number of drafts
bins = pd.cut(df["n_drafts"], [-1, 0, 1, 2, np.inf], labels=["0", "1", "2", "3+"])
acc = df.groupby(bins, observed=True)["correct"].agg(["mean", "size"])
fig, ax = plt.subplots(figsize=(6.5, 4))
bars = ax.bar(acc.index.astype(str), acc["mean"], color=E4B_COLOR, edgecolor="white", linewidth=1.5, width=0.6)
ax.bar_label(bars, labels=[f"{m:.1%}\nn={n}" for m, n in zip(acc["mean"], acc["size"])], fontsize=8, color=MUTED, padding=2)
ax.set_ylim(0, 1.12)
ax.set_xlabel("SQL drafts in the trace")
ax.set_ylabel("share EX-correct")
ax.set_title(f"{E4B}: accuracy by number of drafts (all difficulty levels)", color=INK, fontsize=10, loc="left")
ax.grid(axis="y", color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
fig.tight_layout()
save(fig, "task2_accuracy_by_drafts")